# COMP315 Term Project — Bank Marketing TFX Pipeline
## Phase 1 & 2: All 9 Pipeline Steps

## Cell 1 — Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/COMP315_Project'
os.listdir(PROJECT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


['COMP315_Project_Guide.md',
 'data',
 'notebooks',
 'modules',
 'pipeline',
 'outputs',
 'report',
 'requirements.txt',
 'dags']

## Cell 2 — Full Environment Setup (run once per session)
This installs Python 3.10, creates a virtual environment, and installs TFX with all dependencies pinned to compatible versions.

In [ ]:
# === ENVIRONMENT SETUP (run once per session) ===

# Install Python 3.10 venv
!sudo apt-get update -qq
!sudo apt install python3.10-venv -y -qq
!python3.10 -m venv /content/tfx_env --without-pip
!curl -sS https://bootstrap.pypa.io/get-pip.py | /content/tfx_env/bin/python3.10

# Core TFX packages (pinned, no resolver)
!/content/tfx_env/bin/pip install --no-deps \
  tfx==1.15.0 ml-pipelines-sdk==1.15.0 \
  tensorflow==2.15.0 tensorflow-data-validation==1.15.1 \
  tensorflow-model-analysis==0.46.0 tensorflow-transform==1.15.0 \
  tensorflow-serving-api==2.15.1 tfx-bsl==1.15.1 ml-metadata==1.15.0 \
  tensorflow-metadata==1.15.0 tensorflow-estimator==2.15.0 \
  tensorflow-hub==0.15.0 keras==2.15.0 keras-tuner==1.4.7 kt-legacy==1.0.5

# Dependencies
!/content/tfx_env/bin/pip install --no-deps \
  tensorflow-io-gcs-filesystem==0.37.1 astunparse==1.6.3 flatbuffers==25.12.19 \
  gast==0.7.0 google-pasta==0.2.0 h5py==3.16.0 libclang==18.1.1 \
  ml-dtypes==0.2.0 opt-einsum==3.4.0 termcolor==3.3.0 wrapt==1.14.2 \
  scipy==1.12.0 pandas==1.5.3 joblib==1.5.3 pyfarmhash==0.3.2 \
  pyarrow==10.0.1 pillow==12.3.0 docker==4.4.4 portpicker==1.6.0 \
  psutil==7.2.2 pyyaml==6.0.2 jinja2==3.1.6 click==8.4.2 \
  websocket-client==1.9.0 ipython==7.34.0 ipywidgets==7.8.5 \
  traitlets==5.16.1 widgetsnbextension==3.6.10 jupyterlab-widgets==1.1.11 \
  ipython-genutils==0.2.0 comm==0.2.3 jedi==0.20.0 backcall==0.2.0 \
  pickleshare==0.7.5 pygments==2.20.0 prompt-toolkit==3.0.53 wcwidth==0.8.2 \
  pexpect==4.9.0 ptyprocess==0.7.0 decorator==5.3.1 matplotlib-inline==0.2.2 \
  sacrebleu==2.5.1 rouge-score==0.1.2 tabulate==0.10.0 lxml==6.1.1 \
  colorama==0.4.6 nltk==3.10.2 portalocker==4.1.0

# Pinned infra packages (with deps resolved)
!/content/tfx_env/bin/pip install \
  grpcio==1.62.3 grpcio-status==1.62.3 googleapis-common-protos==1.63.2 \
  proto-plus==1.22.3 protobuf==4.25.9 tensorboard==2.15.2 \
  numpy==1.26.4 apache-beam==2.56.0 absl-py==1.4.0 attrs==23.2.0

# Missing deps from --no-deps installs
!/content/tfx_env/bin/pip install nbformat defusedxml ipykernel \
  google-api-python-client==1.12.11 google-apitools google-auth-httplib2 \
  google-cloud-storage kubernetes==12.0.1 uritemplate

# Fix any version overwrites from the above step
!/content/tfx_env/bin/pip install --no-deps \
  protobuf==4.25.9 grpcio==1.62.3 grpcio-status==1.62.3 \
  googleapis-common-protos==1.63.2 proto-plus==1.22.3 \
  absl-py==1.4.0 attrs==23.2.0 pyyaml==6.0.2

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
python3.10-venv is already the newest version (3.10.12-1~22.04.16).
0 upgraded, 0 newly installed, 0 to remove and 131 not upgraded.
  Using cached pip-26.2.1-py3-none-any.whl.metadata (4.6 kB)
Using cached pip-26.2.1-py3-none-any.whl (1.8 MB)
  Attempting uninstall: pip
    Found existing installation: pip 26.2.1
    Uninstalling pip-26.2.1:
      Successfully uninstalled pip-26.2.1
  Using cached googleapis_common_protos-1.75.1-py3-none-any.whl.metadata (8.5 kB)
  Using cached protobuf-7.35.1-cp310-abi3-manylinux2014_x86_64.whl.metadata (595 bytes)
  Using cached proto_plus-1.28.3-py3-none-any.whl.metadata (2.2 kB)
Using cached googleapis_common_protos-1.75.1-py3-none-any.whl (300 kB)
Using cached proto_plus-1.28.3-py3-none-any.whl (50 kB)
Using cached protobuf-7.35.1-cp310-abi3-manylinux2014_x86_6

## Cell 3 — Verify Installation


In [ ]:
# Verify if tensorflow extended was installed
!/content/tfx_env/bin/python -c "import tfx; print(f'TFX: {tfx.__version__}')"

TFX: 1.15.0


## Cell 4 — Preprocess Data
Converts semicolon-delimited CSV to comma-delimited with numeric labels (yes→1, no→0).

In [ ]:
import pandas as pd
import os

PROJECT_DIR = '/content/drive/MyDrive/COMP315_Project'

df = pd.read_csv(f'{PROJECT_DIR}/data/bank-full.csv', sep=';')
df['y'] = df['y'].map({'yes': 1, 'no': 0})
print(f"Shape: {df.shape}")
print(f"Label distribution:\n{df['y'].value_counts()}")

# Save to separate subfolder so CsvExampleGen only reads this file
os.makedirs(f'{PROJECT_DIR}/data/processed', exist_ok=True)
df.to_csv(f'{PROJECT_DIR}/data/processed/bank_marketing.csv', index=False)
print("Saved to data/processed/bank_marketing.csv")

Shape: (45211, 17)
Label distribution:
y
0    39922
1     5289
Name: count, dtype: int64
Saved to data/processed/bank_marketing.csv


## Cell 5 — Helper Function

In [ ]:
import subprocess

def write_and_run(filename, code):
    """Write code to a file and run it in the TFX Python 3.10 environment."""
    filepath = f'/content/{filename}'
    with open(filepath, 'w') as f:
        f.write(code)
    result = subprocess.run(
        ['/content/tfx_env/bin/python', filepath],
        cwd='/content',
        capture_output=True,
        text=True
    )
    print("STDOUT:")
    print(result.stdout)
    if result.returncode != 0:
        print("\nERROR:")
        print(result.stderr[-2000:])
    else:
        print("\nSUCCESS")
    return result.returncode

## Cell 6 — Phase 1: Steps 1–4 (ExampleGen → StatisticsGen → SchemaGen → ExampleValidator)

In [ ]:
PROJECT_DIR = '/content/drive/MyDrive/COMP315_Project'

write_and_run('run_phase1.py', f'''
import os
import tensorflow as tf
import tfx
from tfx.orchestration.local.local_dag_runner import LocalDagRunner
from tfx.orchestration import pipeline as pipeline_module
from tfx.components import CsvExampleGen, StatisticsGen, SchemaGen, ExampleValidator

_pipeline_name = "bank_marketing_pipeline"
_pipeline_root = "/content/pipelines/" + _pipeline_name
_metadata_path = "/content/metadata/" + _pipeline_name + "/metadata.db"
_data_root = "{PROJECT_DIR}/data/processed"

for d in [_pipeline_root, os.path.dirname(_metadata_path)]:
    os.makedirs(d, exist_ok=True)

print(f"TFX: {{tfx.__version__}}")
print(f"TF:  {{tf.__version__}}")

example_gen = CsvExampleGen(input_base=_data_root)
statistics_gen = StatisticsGen(examples=example_gen.outputs["examples"])
schema_gen = SchemaGen(statistics=statistics_gen.outputs["statistics"])
example_validator = ExampleValidator(
    statistics=statistics_gen.outputs["statistics"],
    schema=schema_gen.outputs["schema"]
)

components = [example_gen, statistics_gen, schema_gen, example_validator]

p = pipeline_module.Pipeline(
    pipeline_name=_pipeline_name,
    pipeline_root=_pipeline_root,
    components=components,
    metadata_connection_config=tfx.orchestration.metadata.sqlite_metadata_connection_config(_metadata_path),
)

print("Running Phase 1 pipeline...")
LocalDagRunner().run(p)
print("\\n=== PHASE 1 COMPLETE ===")

from ml_metadata.metadata_store import metadata_store
from ml_metadata.proto import metadata_store_pb2
connection_config = metadata_store_pb2.ConnectionConfig()
connection_config.sqlite.filename_uri = _metadata_path
store = metadata_store.MetadataStore(connection_config)
for artifact_type in store.get_artifact_types():
    artifacts = store.get_artifacts_by_type(artifact_type.name)
    if artifacts:
        print(f"\\n{{artifact_type.name}} ({{len(artifacts)}} artifacts):")
        for a in artifacts[-2:]:
            print(f"  URI: {{a.uri}}")
''')

STDOUT:
TFX: 1.15.0
TF:  2.15.0
Running Phase 1 pipeline...

=== PHASE 1 COMPLETE ===

Examples (8 artifacts):
  URI: /content/pipelines/bank_marketing_pipeline/Transform/transformed_examples/28
  URI: /content/pipelines/bank_marketing_pipeline/CsvExampleGen/examples/33

ExampleStatistics (11 artifacts):
  URI: /content/pipelines/bank_marketing_pipeline/Transform/post_transform_stats/28
  URI: /content/pipelines/bank_marketing_pipeline/StatisticsGen/statistics/34

Schema (11 artifacts):
  URI: /content/pipelines/bank_marketing_pipeline/Transform/pre_transform_schema/28
  URI: /content/pipelines/bank_marketing_pipeline/SchemaGen/schema/35

ExampleAnomalies (8 artifacts):
  URI: /content/pipelines/bank_marketing_pipeline/Transform/post_transform_anomalies/28
  URI: /content/pipelines/bank_marketing_pipeline/ExampleValidator/anomalies/36

TransformGraph (3 artifacts):
  URI: /content/pipelines/bank_marketing_pipeline/Transform/transform_graph/19
  URI: /content/pipelines/bank_marketing_pi

0

## Cell 7 — Phase 2 Run 1: Steps 1–9 (500 train steps)
Steps 1–4 use cached results from Phase 1. Steps 5–9 are new.

In [51]:
PROJECT_DIR = '/content/drive/MyDrive/COMP315_Project'

write_and_run('run_phase2.py', f'''
import os
import tensorflow as tf
import tfx
from tfx.orchestration.local.local_dag_runner import LocalDagRunner
from tfx.orchestration import pipeline as pipeline_module
from tfx.components import (
    CsvExampleGen, StatisticsGen, SchemaGen, ExampleValidator,
    Transform, Trainer, Evaluator, Pusher
)
from tfx.dsl.components.common.resolver import Resolver
from tfx.proto import trainer_pb2, pusher_pb2
from tfx.types import Channel
from tfx.types.standard_artifacts import Model, ModelBlessing
from tfx.dsl.input_resolution.strategies.latest_blessed_model_strategy import LatestBlessedModelStrategy
import tensorflow_model_analysis as tfma

_pipeline_name = "bank_marketing_pipeline"
_pipeline_root = "/content/pipelines/" + _pipeline_name
_metadata_path = "/content/metadata/" + _pipeline_name + "/metadata.db"
_data_root = "{PROJECT_DIR}/data/processed"
_serving_model_dir = "/content/serving_model/" + _pipeline_name
_transform_module = "{PROJECT_DIR}/modules/transform_module.py"
_trainer_module = "{PROJECT_DIR}/modules/trainer_module.py"

for d in [_pipeline_root, os.path.dirname(_metadata_path), _serving_model_dir]:
    os.makedirs(d, exist_ok=True)

print(f"TFX: {{tfx.__version__}}")
print(f"TF:  {{tf.__version__}}")

# Step 1
example_gen = CsvExampleGen(input_base=_data_root)
# Step 2
statistics_gen = StatisticsGen(examples=example_gen.outputs["examples"])
# Step 3
schema_gen = SchemaGen(statistics=statistics_gen.outputs["statistics"])
# Step 4
example_validator = ExampleValidator(
    statistics=statistics_gen.outputs["statistics"],
    schema=schema_gen.outputs["schema"]
)
# Step 5: Transform
transform = Transform(
    examples=example_gen.outputs["examples"],
    schema=schema_gen.outputs["schema"],
    module_file=_transform_module
)
# Step 6: Trainer (500 steps)
trainer = Trainer(
    module_file=_trainer_module,
    examples=transform.outputs["transformed_examples"],
    transform_graph=transform.outputs["transform_graph"],
    schema=schema_gen.outputs["schema"],
    train_args=trainer_pb2.TrainArgs(num_steps=500),
    eval_args=trainer_pb2.EvalArgs(num_steps=100)
)
# Step 7: Resolver
model_resolver = Resolver(
    strategy_class=LatestBlessedModelStrategy,
    model=Channel(type=Model),
    model_blessing=Channel(type=ModelBlessing)
).with_id("latest_blessed_model_resolver")
# Step 8: Evaluator
eval_config = tfma.EvalConfig(
    model_specs=[tfma.ModelSpec(label_key="y")],
    slicing_specs=[
        tfma.SlicingSpec(),
        tfma.SlicingSpec(feature_keys=["education"]),
        tfma.SlicingSpec(feature_keys=["marital"]),
        tfma.SlicingSpec(feature_keys=["job"]),
    ],
    metrics_specs=[
        tfma.MetricsSpec(metrics=[
            tfma.MetricConfig(class_name="BinaryAccuracy"),
            tfma.MetricConfig(class_name="AUC"),
            tfma.MetricConfig(class_name="ExampleCount"),
        ])
    ]
)
evaluator = Evaluator(
    examples=example_gen.outputs["examples"],
    model=trainer.outputs["model"],
    baseline_model=model_resolver.outputs["model"],
    eval_config=eval_config
)
# Step 9: Pusher
pusher = Pusher(
    model=trainer.outputs["model"],
    model_blessing=evaluator.outputs["blessing"],
    push_destination=pusher_pb2.PushDestination(
        filesystem=pusher_pb2.PushDestination.Filesystem(
            base_directory=_serving_model_dir
        )
    )
)

components = [
    example_gen, statistics_gen, schema_gen, example_validator,
    transform, trainer, model_resolver, evaluator, pusher
]

p = pipeline_module.Pipeline(
    pipeline_name=_pipeline_name,
    pipeline_root=_pipeline_root,
    components=components,
    metadata_connection_config=tfx.orchestration.metadata.sqlite_metadata_connection_config(_metadata_path),
)

print("Running full pipeline RUN 1 (500 steps)...")
LocalDagRunner().run(p)
print("\\n=== RUN 1 COMPLETE (ALL 9 STEPS) ===")

from ml_metadata.metadata_store import metadata_store
from ml_metadata.proto import metadata_store_pb2
connection_config = metadata_store_pb2.ConnectionConfig()
connection_config.sqlite.filename_uri = _metadata_path
store = metadata_store.MetadataStore(connection_config)
for artifact_type in store.get_artifact_types():
    artifacts = store.get_artifacts_by_type(artifact_type.name)
    if artifacts:
        print(f"\\n{{artifact_type.name}} ({{len(artifacts)}} artifacts):")
        for a in artifacts[-2:]:
            print(f"  URI: {{a.uri}}")

import shutil
artifacts_dir = "{PROJECT_DIR}/outputs/artifacts"
os.makedirs(artifacts_dir, exist_ok=True)
for a in store.get_artifacts_by_type("ModelEvaluation"):
    eval_dst = os.path.join(artifacts_dir, "eval_run_1")
    shutil.copytree(a.uri, eval_dst, dirs_exist_ok=True)
    print(f"\\nEvaluation saved to: {{eval_dst}}")
for a in store.get_artifacts_by_type("ModelRun"):
    run_dst = os.path.join(artifacts_dir, "model_run_1")
    shutil.copytree(a.uri, run_dst, dirs_exist_ok=True)
    print(f"TensorBoard logs saved to: {{run_dst}}")

print("\\nRun 1 artifacts saved to Drive.")
''')

STDOUT:
running bdist_wheel
running build
running build_py
creating build/lib
copying transform_module.py -> build/lib
copying trainer_module.py -> build/lib
installing to /tmp/tmpl6txx089
running install
running install_lib
copying build/lib/trainer_module.py -> /tmp/tmpl6txx089/.
copying build/lib/transform_module.py -> /tmp/tmpl6txx089/.
running install_egg_info
running egg_info
creating tfx_user_code_Transform.egg-info
writing tfx_user_code_Transform.egg-info/PKG-INFO
writing dependency_links to tfx_user_code_Transform.egg-info/dependency_links.txt
writing top-level names to tfx_user_code_Transform.egg-info/top_level.txt
writing manifest file 'tfx_user_code_Transform.egg-info/SOURCES.txt'
reading manifest file 'tfx_user_code_Transform.egg-info/SOURCES.txt'
writing manifest file 'tfx_user_code_Transform.egg-info/SOURCES.txt'
Copying tfx_user_code_Transform.egg-info to /tmp/tmpl6txx089/./tfx_user_code_Transform-0.0+f8ad0b05869a2295c5df003699658ba269b5e95ca2fa20adcf38774eed48d34c-py3.

0

## Cell 8 — Phase 2 Run 2: Steps 1–9 (1000 train steps)
Same pipeline, doubled training steps. Needed for TFMA two-run comparison.

In [ ]:
PROJECT_DIR = '/content/drive/MyDrive/COMP315_Project'

write_and_run('run_phase2_v2.py', f'''
import os
import tensorflow as tf
import tfx
from tfx.orchestration.local.local_dag_runner import LocalDagRunner
from tfx.orchestration import pipeline as pipeline_module
from tfx.components import (
    CsvExampleGen, StatisticsGen, SchemaGen, ExampleValidator,
    Transform, Trainer, Evaluator, Pusher
)
from tfx.dsl.components.common.resolver import Resolver
from tfx.proto import trainer_pb2, pusher_pb2
from tfx.types import Channel
from tfx.types.standard_artifacts import Model, ModelBlessing
from tfx.dsl.input_resolution.strategies.latest_blessed_model_strategy import LatestBlessedModelStrategy
import tensorflow_model_analysis as tfma

_pipeline_name = "bank_marketing_pipeline"
_pipeline_root = "/content/pipelines/" + _pipeline_name
_metadata_path = "/content/metadata/" + _pipeline_name + "/metadata.db"
_data_root = "{PROJECT_DIR}/data/processed"
_serving_model_dir = "/content/serving_model/" + _pipeline_name
_transform_module = "{PROJECT_DIR}/modules/transform_module.py"
_trainer_module = "{PROJECT_DIR}/modules/trainer_module.py"

for d in [_pipeline_root, os.path.dirname(_metadata_path), _serving_model_dir]:
    os.makedirs(d, exist_ok=True)

example_gen = CsvExampleGen(input_base=_data_root)
statistics_gen = StatisticsGen(examples=example_gen.outputs["examples"])
schema_gen = SchemaGen(statistics=statistics_gen.outputs["statistics"])
example_validator = ExampleValidator(
    statistics=statistics_gen.outputs["statistics"],
    schema=schema_gen.outputs["schema"]
)
transform = Transform(
    examples=example_gen.outputs["examples"],
    schema=schema_gen.outputs["schema"],
    module_file=_transform_module
)
# CHANGED: 500 -> 1000 train steps, 100 -> 200 eval steps
trainer = Trainer(
    module_file=_trainer_module,
    examples=transform.outputs["transformed_examples"],
    transform_graph=transform.outputs["transform_graph"],
    schema=schema_gen.outputs["schema"],
    train_args=trainer_pb2.TrainArgs(num_steps=1000),
    eval_args=trainer_pb2.EvalArgs(num_steps=200)
)
model_resolver = Resolver(
    strategy_class=LatestBlessedModelStrategy,
    model=Channel(type=Model),
    model_blessing=Channel(type=ModelBlessing)
).with_id("latest_blessed_model_resolver")
eval_config = tfma.EvalConfig(
    model_specs=[tfma.ModelSpec(label_key="y")],
    slicing_specs=[
        tfma.SlicingSpec(),
        tfma.SlicingSpec(feature_keys=["education"]),
        tfma.SlicingSpec(feature_keys=["marital"]),
        tfma.SlicingSpec(feature_keys=["job"]),
    ],
    metrics_specs=[
        tfma.MetricsSpec(metrics=[
            tfma.MetricConfig(class_name="BinaryAccuracy"),
            tfma.MetricConfig(class_name="AUC"),
            tfma.MetricConfig(class_name="ExampleCount"),
        ])
    ]
)
evaluator = Evaluator(
    examples=example_gen.outputs["examples"],
    model=trainer.outputs["model"],
    baseline_model=model_resolver.outputs["model"],
    eval_config=eval_config
)
pusher = Pusher(
    model=trainer.outputs["model"],
    model_blessing=evaluator.outputs["blessing"],
    push_destination=pusher_pb2.PushDestination(
        filesystem=pusher_pb2.PushDestination.Filesystem(
            base_directory=_serving_model_dir
        )
    )
)

components = [
    example_gen, statistics_gen, schema_gen, example_validator,
    transform, trainer, model_resolver, evaluator, pusher
]

p = pipeline_module.Pipeline(
    pipeline_name=_pipeline_name,
    pipeline_root=_pipeline_root,
    components=components,
    metadata_connection_config=tfx.orchestration.metadata.sqlite_metadata_connection_config(_metadata_path),
)

print("Running pipeline RUN 2 (1000 steps)...")
LocalDagRunner().run(p)
print("\\n=== RUN 2 COMPLETE ===")

from ml_metadata.metadata_store import metadata_store
from ml_metadata.proto import metadata_store_pb2
connection_config = metadata_store_pb2.ConnectionConfig()
connection_config.sqlite.filename_uri = _metadata_path
store = metadata_store.MetadataStore(connection_config)
for artifact_type in store.get_artifact_types():
    artifacts = store.get_artifacts_by_type(artifact_type.name)
    if artifacts:
        print(f"\\n{{artifact_type.name}} ({{len(artifacts)}} artifacts):")
        for a in artifacts[-1:]:
            print(f"  URI: {{a.uri}}")

import shutil
artifacts_dir = "{PROJECT_DIR}/outputs/artifacts"
# CHANGED: save as run_2
for a in store.get_artifacts_by_type("ModelEvaluation"):
    eval_dst = os.path.join(artifacts_dir, "eval_run_2")
    shutil.copytree(a.uri, eval_dst, dirs_exist_ok=True)
    print(f"\\nEvaluation saved to: {{eval_dst}}")
for a in store.get_artifacts_by_type("ModelRun"):
    run_dst = os.path.join(artifacts_dir, "model_run_2")
    shutil.copytree(a.uri, run_dst, dirs_exist_ok=True)
    print(f"TensorBoard logs saved to: {{run_dst}}")

print("\\nRun 2 artifacts saved to Drive.")
''')

STDOUT:
running bdist_wheel
running build
running build_py
creating build/lib
copying transform_module.py -> build/lib
copying trainer_module.py -> build/lib
installing to /tmp/tmpuu15moon
running install
running install_lib
copying build/lib/trainer_module.py -> /tmp/tmpuu15moon/.
copying build/lib/transform_module.py -> /tmp/tmpuu15moon/.
running install_egg_info
running egg_info
creating tfx_user_code_Transform.egg-info
writing tfx_user_code_Transform.egg-info/PKG-INFO
writing dependency_links to tfx_user_code_Transform.egg-info/dependency_links.txt
writing top-level names to tfx_user_code_Transform.egg-info/top_level.txt
writing manifest file 'tfx_user_code_Transform.egg-info/SOURCES.txt'
reading manifest file 'tfx_user_code_Transform.egg-info/SOURCES.txt'
writing manifest file 'tfx_user_code_Transform.egg-info/SOURCES.txt'
Copying tfx_user_code_Transform.egg-info to /tmp/tmpuu15moon/./tfx_user_code_Transform-0.0+f8ad0b05869a2295c5df003699658ba269b5e95ca2fa20adcf38774eed48d34c-py3.

0

# Phase 3: Airflow DAG Setup & Screenshots
**Prerequisites:** Run the Phase 1 & 2 notebook first (Cells 1-7 minimum) so TFX is installed and the pipeline has run at least once.

This notebook sets up Airflow in the same Python 3.10 venv, creates a DAG from your TFX pipeline, and exposes the Airflow UI so you can trigger runs and take screenshots.

## Cell 9 — Install Airflow 2.7.3 (compatible with TFX 1.15.0)
Airflow 3.x was installed earlier and is NOT compatible with TFX's AirflowDagRunner. We install 2.7.3 specifically.

In [ ]:
# Install Airflow 2.7.3 (compatible with TFX 1.15.0)
!/content/tfx_env/bin/pip install apache-airflow==2.7.3 \
  --constraint "https://raw.githubusercontent.com/apache/airflow/constraints-2.7.3/constraints-3.10.txt" \
  2>&1 | tail -5

# Fix any overwrites
!/content/tfx_env/bin/pip install --no-deps \
  protobuf==4.25.9 grpcio==1.62.3 grpcio-status==1.62.3 \
  googleapis-common-protos==1.63.2 proto-plus==1.22.3 \
  absl-py==1.4.0 attrs==23.2.0 pyyaml==6.0.2 2>&1 | tail -3

# Verify both still work
!/content/tfx_env/bin/python -c "import tfx, airflow; print(f'TFX: {tfx.__version__}'); print(f'Airflow: {airflow.__version__}')"

tensorflow 2.15.0 requires wrapt<1.15,>=1.11.0, but you have wrapt 1.15.0 which is incompatible.
tensorflow-metadata 1.15.0 requires protobuf<4.21,>=3.20.3; python_version < "3.11", but you have protobuf 4.24.4 which is incompatible.
tensorflow-serving-api 2.15.1 requires tensorflow<3,>=2.15.1, but you have tensorflow 2.15.0 which is incompatible.
wheel 0.47.0 requires packaging>=24.0, but you have packaging 23.2 which is incompatible.
      Successfully uninstalled attrs-23.1.0

TFX: 1.15.0
Airflow: 2.7.3


## Cell 10 — Initialize Airflow & Create DAG File
Sets up the Airflow database, creates an admin user, and writes the TFX pipeline DAG.

In [ ]:
import os
import subprocess

AIRFLOW_HOME = '/content/airflow'
os.environ['AIRFLOW_HOME'] = AIRFLOW_HOME
os.makedirs(f'{AIRFLOW_HOME}/dags', exist_ok=True)

env = os.environ.copy()
env['AIRFLOW_HOME'] = AIRFLOW_HOME
env['AIRFLOW__CORE__LOAD_EXAMPLES'] = 'False'
env['AIRFLOW__CORE__DAGS_FOLDER'] = f'{AIRFLOW_HOME}/dags'
env['AIRFLOW__DATABASE__SQL_ALCHEMY_CONN'] = f'sqlite:///{AIRFLOW_HOME}/airflow.db'

# Initialize the Airflow metadata database
print("Initializing Airflow DB...")
subprocess.run(
    ['/content/tfx_env/bin/airflow', 'db', 'init'],
    env=env, capture_output=True
)

# Create admin user
print("Creating admin user...")
subprocess.run(
    ['/content/tfx_env/bin/airflow', 'users', 'create',
     '--username', 'admin', '--password', 'admin',
     '--firstname', 'Admin', '--lastname', 'User',
     '--role', 'Admin', '--email', 'admin@example.com'],
    env=env, capture_output=True
)

# Write the DAG file
PROJECT_DIR = '/content/drive/MyDrive/COMP315_Project'

dag_code = f'''
import os
import sys
from datetime import datetime

# Ensure modules are importable
sys.path.insert(0, "{PROJECT_DIR}/modules")

from airflow import DAG
from tfx.components import (
    CsvExampleGen, StatisticsGen, SchemaGen, ExampleValidator,
    Transform, Trainer, Evaluator, Pusher
)
from tfx.dsl.components.common.resolver import Resolver
from tfx.proto import trainer_pb2, pusher_pb2
from tfx.types import Channel
from tfx.types.standard_artifacts import Model, ModelBlessing
from tfx.dsl.input_resolution.strategies.latest_blessed_model_strategy import LatestBlessedModelStrategy
from tfx.orchestration import pipeline as pipeline_module
from tfx.orchestration.airflow.airflow_dag_runner import AirflowDagRunner, AirflowPipelineConfig
import tensorflow_model_analysis as tfma

_pipeline_name = "bank_marketing_pipeline"
_pipeline_root = "/content/airflow/pipelines/" + _pipeline_name
_metadata_path = "/content/airflow/metadata/" + _pipeline_name + "/metadata.db"
_data_root = "{PROJECT_DIR}/data/processed"
_serving_model_dir = "/content/airflow/serving_model/" + _pipeline_name
_transform_module = "{PROJECT_DIR}/modules/transform_module.py"
_trainer_module = "{PROJECT_DIR}/modules/trainer_module.py"

for d in [_pipeline_root, os.path.dirname(_metadata_path), _serving_model_dir]:
    os.makedirs(d, exist_ok=True)

# Step 1
example_gen = CsvExampleGen(input_base=_data_root)
# Step 2
statistics_gen = StatisticsGen(examples=example_gen.outputs["examples"])
# Step 3
schema_gen = SchemaGen(statistics=statistics_gen.outputs["statistics"])
# Step 4
example_validator = ExampleValidator(
    statistics=statistics_gen.outputs["statistics"],
    schema=schema_gen.outputs["schema"]
)
# Step 5
transform = Transform(
    examples=example_gen.outputs["examples"],
    schema=schema_gen.outputs["schema"],
    module_file=_transform_module
)
# Step 6
trainer = Trainer(
    module_file=_trainer_module,
    examples=transform.outputs["transformed_examples"],
    transform_graph=transform.outputs["transform_graph"],
    schema=schema_gen.outputs["schema"],
    train_args=trainer_pb2.TrainArgs(num_steps=100),
    eval_args=trainer_pb2.EvalArgs(num_steps=50)
)
# Step 7
model_resolver = Resolver(
    strategy_class=LatestBlessedModelStrategy,
    model=Channel(type=Model),
    model_blessing=Channel(type=ModelBlessing)
).with_id("latest_blessed_model_resolver")
# Step 8
eval_config = tfma.EvalConfig(
    model_specs=[tfma.ModelSpec(label_key="y")],
    slicing_specs=[
        tfma.SlicingSpec(),
        tfma.SlicingSpec(feature_keys=["education"]),
        tfma.SlicingSpec(feature_keys=["marital"]),
    ],
    metrics_specs=[
        tfma.MetricsSpec(metrics=[
            tfma.MetricConfig(class_name="BinaryAccuracy"),
            tfma.MetricConfig(class_name="AUC"),
        ])
    ]
)
evaluator = Evaluator(
    examples=example_gen.outputs["examples"],
    model=trainer.outputs["model"],
    baseline_model=model_resolver.outputs["model"],
    eval_config=eval_config
)
# Step 9
pusher = Pusher(
    model=trainer.outputs["model"],
    model_blessing=evaluator.outputs["blessing"],
    push_destination=pusher_pb2.PushDestination(
        filesystem=pusher_pb2.PushDestination.Filesystem(
            base_directory=_serving_model_dir
        )
    )
)

components = [
    example_gen, statistics_gen, schema_gen, example_validator,
    transform, trainer, model_resolver, evaluator, pusher
]

tfx_pipeline = pipeline_module.Pipeline(
    pipeline_name=_pipeline_name,
    pipeline_root=_pipeline_root,
    components=components,
    metadata_connection_config=(
        tfx.orchestration.metadata.sqlite_metadata_connection_config(_metadata_path)
    ),
)

airflow_config = {{
    "schedule_interval": None,
    "start_date": datetime(2024, 1, 1),
    "catchup": False,
}}

DAG = AirflowDagRunner(
    AirflowPipelineConfig(airflow_dag_config=airflow_config)
).run(tfx_pipeline)
'''

with open(f'{AIRFLOW_HOME}/dags/bank_marketing_dag.py', 'w') as f:
    f.write(dag_code)

print(f"DAG file written to {AIRFLOW_HOME}/dags/bank_marketing_dag.py")
print(f"Airflow home: {AIRFLOW_HOME}")
print("Setup complete.")

Initializing Airflow DB...
Creating admin user...
DAG file written to /content/airflow/dags/bank_marketing_dag.py
Airflow home: /content/airflow
Setup complete.


In [32]:
!/content/tfx_env/bin/pip install "jupyter_client<8" "jupyter_core<6"

  Attempting uninstall: jupyter_client
    Found existing installation: jupyter_client 8.9.1
    Uninstalling jupyter_client-8.9.1:
      Successfully uninstalled jupyter_client-8.9.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [jupyter_client]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipykernel 7.3.0 requires jupyter-client>=8.9.0, but you have jupyter-client 7.4.9 which is incompatible.


In [34]:
# Read the current DAG file and add the missing import
dag_path = '/content/airflow/dags/bank_marketing_dag.py'

with open(dag_path, 'r') as f:
    content = f.read()

# Add 'import tfx' after 'from airflow import DAG'
content = content.replace(
    'from airflow import DAG',
    'from airflow import DAG\nimport tfx'
)

with open(dag_path, 'w') as f:
    f.write(content)

print("Fixed: added 'import tfx' to DAG file")

Fixed: added 'import tfx' to DAG file


## Cell 11 — Verify DAG Loads
Check that Airflow can parse the DAG file without errors before starting the webserver.

In [35]:
import subprocess, os

env = os.environ.copy()
env['AIRFLOW_HOME'] = '/content/airflow'
env['AIRFLOW__CORE__LOAD_EXAMPLES'] = 'False'
env['AIRFLOW__CORE__DAGS_FOLDER'] = '/content/airflow/dags'
env['AIRFLOW__DATABASE__SQL_ALCHEMY_CONN'] = 'sqlite:////content/airflow/airflow.db'

result = subprocess.run(
    ['/content/tfx_env/bin/airflow', 'dags', 'list'],
    env=env, capture_output=True, text=True
)
print(result.stdout)
if 'bank_marketing_pipeline' in result.stdout:
    print("DAG loaded successfully!")
else:
    # Check for errors
    result2 = subprocess.run(
        ['/content/tfx_env/bin/airflow', 'dags', 'list-import-errors'],
        env=env, capture_output=True, text=True
    )
    print("Import errors:")
    print(result2.stdout)

running bdist_wheel
running build
running build_py
creating build/lib
copying transform_module.py -> build/lib
copying trainer_module.py -> build/lib
installing to /tmp/tmpk2aqlhum
running install
running install_lib
copying build/lib/trainer_module.py -> /tmp/tmpk2aqlhum/.
copying build/lib/transform_module.py -> /tmp/tmpk2aqlhum/.
running install_egg_info
running egg_info
creating tfx_user_code_Transform.egg-info
writing tfx_user_code_Transform.egg-info/PKG-INFO
writing dependency_links to tfx_user_code_Transform.egg-info/dependency_links.txt
writing top-level names to tfx_user_code_Transform.egg-info/top_level.txt
writing manifest file 'tfx_user_code_Transform.egg-info/SOURCES.txt'
reading manifest file 'tfx_user_code_Transform.egg-info/SOURCES.txt'
writing manifest file 'tfx_user_code_Transform.egg-info/SOURCES.txt'
Copying tfx_user_code_Transform.egg-info to /tmp/tmpk2aqlhum/./tfx_user_code_Transform-0.0+f8ad0b05869a2295c5df003699658ba269b5e95ca2fa20adcf38774eed48d34c-py3.10.egg-i

## Cell 12 — Start Airflow Webserver & Scheduler
Starts both in the background. The webserver runs on port 8080.

In [50]:
import subprocess, os

env = os.environ.copy()
env['AIRFLOW_HOME'] = '/content/airflow'
env['AIRFLOW__CORE__LOAD_EXAMPLES'] = 'False'
env['AIRFLOW__CORE__DAGS_FOLDER'] = '/content/airflow/dags'
env['AIRFLOW__DATABASE__SQL_ALCHEMY_CONN'] = 'sqlite:////content/airflow/airflow.db'

# Kill any existing Airflow processes
!pkill -f 'airflow' 2>/dev/null
!pkill -f 'gunicorn' 2>/dev/null
!rm -f /content/airflow/airflow-webserver.pid

# === SCREENSHOT 1: DAG Graph ===
# Generate DAG graph as an image file
print("Generating DAG graph image...")
result = subprocess.run(
    ['/content/tfx_env/bin/airflow', 'dags', 'show', 'bank_marketing_pipeline',
     '--save', '/content/drive/MyDrive/COMP315_Project/outputs/screenshots/airflow_dag_graph.png'],
    env=env, capture_output=True, text=True
)
if result.returncode == 0:
    print("✅ DAG graph saved to outputs/screenshots/airflow_dag_graph.png")
else:
    print(f"Error: {result.stderr[-500:]}")

# === SCREENSHOT 2 & 3: Run the DAG and capture logs ===
print("\nRunning DAG (this takes ~10 min)...")
result = subprocess.run(
    ['/content/tfx_env/bin/airflow', 'dags', 'test', 'bank_marketing_pipeline', '2024-01-01'],
    env=env, capture_output=True, text=True, timeout=900
)

print("STDOUT (last 3000 chars):")
print(result.stdout[-3000:])

if result.returncode == 0:
    print("\n✅ DAG RUN SUCCESSFUL")
else:
    print(f"\nERROR: {result.stderr[-1500:]}")

# Save the full output as a text file for the report
with open('/content/drive/MyDrive/COMP315_Project/outputs/screenshots/airflow_dag_run_log.txt', 'w') as f:
    f.write("=== DAG TEST RUN OUTPUT ===\n\n")
    f.write("STDOUT:\n")
    f.write(result.stdout)
    f.write("\n\nSTDERR:\n")
    f.write(result.stderr)
print("Full log saved to outputs/screenshots/airflow_dag_run_log.txt")

^C
^C
Generating DAG graph image...
✅ DAG graph saved to outputs/screenshots/airflow_dag_graph.png

Running DAG (this takes ~10 min)...
STDOUT (last 3000 chars):
260809T180414
[2026-08-09T18:04:14.343+0000] {taskinstance.py:1400} INFO - Marking task as SUCCESS. dag_id=bank_marketing_pipeline, task_id=Evaluator, execution_date=20240101T000000, start_date=, end_date=20260809T180414
[2026-08-09T18:04:14.356+0000] {dag.py:3938} INFO - Evaluator ran successfully!
[2026-08-09T18:04:14.357+0000] {dag.py:3941} INFO - *****************************************************
[2026-08-09T18:04:14.362+0000] {dag.py:3930} INFO - *****************************************************
[2026-08-09T18:04:14.362+0000] {dag.py:3934} INFO - Running task Pusher
[2026-08-09 18:04:14,400] {taskinstance.py:1662} INFO - Exporting env vars: AIRFLOW_CTX_DAG_OWNER='airflow' AIRFLOW_CTX_DAG_ID='bank_marketing_pipeline' AIRFLOW_CTX_TASK_ID='Pusher' AIRFLOW_CTX_EXECUTION_DATE='2024-01-01T00:00:00+00:00' AIRFLOW_CTX_TRY_